# Tutorial 2: Differential Spatial Expression (DSE) Analysis in Mouse Brain (Multi-Replicate)

In this tutorial, we will conduct a Differential Spatial Expression (DSE) analysis across **two groups with multiple replicates** on a mouse hypothalamic preoptic region dataset. This example will demonstrate how to preprocess the data and identify differentially spatially expressed genes across group. This tutorial, we include cell type as an optional covariate to account for the potential confounding effect that varying cell types may have on spatial expression patterns. 

*Dataset Reference*: Jeffrey R. Moffitt et al., *Molecular, spatial, and functional single-cell profiling of the hypothalamic preoptic region*, *Science*, 362, eaau5324 (2018). DOI: 10.1126/science.aau5324

**Note**: Ensure all necessary libraries are installed, and you have a compatible environment set up before running the code cells. Expected run time for the tutorial is approximately 2 hours and 30 minutes, depending on the system's performance and available resources.

If you have any questions, please contact Yeojin Kim at ykim3030@gatech.edu.

## Preprocessing

Apply sample-specific filtering, normalization, and log transformation first.
`se.preprocessing` then selects common genes, zeros pooled mean+4SD outliers,
and takes the union of 1,000 HVGs per sample. All common genes are retained
when fewer than 1,000 are available.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
import scanpy as sc
import pandas as pd
import igraph
import pickle

In [ ]:
import SpaceExpress as se

The original data can be downloaded from [here](https://datadryad.org/stash/dataset/doi:10.5061/dryad.8t8s248). Check the [`Tutorial-data`](./Tutorial_data.ipynb) for generating anndata h5ad file from the original dataset. We selected sections of naïve mice (ID 5, 6, 7) and pup-exposed mice (ID 34, 35, 36) at bregma 0.16 mm for the analysis. Each sample contains 4,908-5,998 cells, respectively. The expression values for each section are stored in an AnnData h5ad format, along with the associated metadata for each sample. The location of each cell is stored in the obsm['spatial'] column.

In [ ]:
# Load the data 
file_list = ['5', '6', '7', '34', '35', '36'] # 5,6,7: Naive, 34,35,36: Pup-exposed
adata_list = [sc.read_h5ad(f'./Tutorial_data/MERFISH-mouse-{f}.h5ad') for f in file_list]

In [ ]:
# Normalization
for i, adata in enumerate(adata_list):
    genes_to_keep = [gene for gene in adata.var_names if 'Blank' not in gene]
    adata = adata[:, genes_to_keep].copy() 
    sc.pp.filter_genes(adata, min_cells=3)
    sc.pp.filter_cells(adata, min_genes=3)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata_list[i] = adata

adata_list = se.preprocessing(adata_list)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(file_list), figsize=(len(file_list)*4, 3))  # Adjust the figure size as needed

for i in range(len(adata_list)):
    adata = adata_list[i]
    ax = axes[i]
    plt.rcParams["font.size"] = 10
    sc.pl.embedding(adata, basis="spatial", s=15, show=False, ax=ax)
    ax.set_title(file_list[i])
    ax.axis('off')

plt.tight_layout()
plt.show()

## 1) Constructing the k-Nearest Neighbor Graph

In this step, we'll build a k-nearest neighbor (k-NN) graph to capture local spatial relationships among cells. This graph forms a critical component of the spatial embedding in SpaceExpress, as it defines neighboring relationships that are used for downstream spatial analysis.

- **Choosing k**: The `choose_k` function helps to select an optimal number of neighbors for the k-NN graph. While you can set this value manually, we recommend using `choose_k` for data-driven selection. The function will identify the smallest k that results in the fewest disconnected components in the graph. If k < 20 does not yield a fully connected graph, the function will recommend the smallest k that minimizes disconnected components. 

In [ ]:
# Step 1: Automatically select the optimal k for the entire dataset
k = se.choose_k(adata_list)

In this case, for six different samples, we will manually set `k`=4, as this is the minimum `k` required to ensure the k-NN graph is fully connected in most cases, allowing us to better infer the underlying spatial structure. In the multi-replicate case, we recommend setting `k` based on the minimum value that ensures a fully connected k-NN graph for the majority of the samples.

In [ ]:
k = 4

Once k is chosen, we proceed to build the k-NN graph for each dataset in the analysis.

- **shortest_path**: The `shortest_path` function generates a shortest path distance matrix from the k-NN graph, capturing spatial distances between neighboring cells. If the k-NN graph of a sample contains disconnected components, this function adds the minimum number of edges required to ensure a fully connected graph by linking isolated clusters or nodes to their nearest neighbors. This step is essential for accurately calculating spatial relationships between cells.

In [ ]:
!mkdir -p ./Tutorial_data/Tutorial_2_data

In [ ]:
# Step 2: Construct and save the k-nearest neighbor graph for each dataset
for i in range(len(adata_list)):
    adata = adata_list[i]
    se.shortest_path(adata.obsm['spatial'], f'Tutorial_data/Tutorial_2_data/MERFISH-mouse-{file_list[i]}-shortest-paths.pkl', k = k)

## 2) Round 1 of DSE Gene Detection

In this section, we initiate the first round of Differential Spatial Expression (DSE) gene detection.

In [ ]:
# Step 1: Define the list of shortest path files for each sample
shortest_file_path_list = [f'Tutorial_data/Tutorial_2_data/MERFISH-mouse-{i}-shortest-paths.pkl' for i in file_list]

# Step 2: Train the SpaceExpress model
embedding_before_filtered = se.train_SpaceExpress_multi(
    adata_list, 
    shortest_file_path_list, 
    epochs = 100
)

The `SpaceExpress_DSE` function supports multi-threading to speed up computation. For these 6 mouse brain datasets, which contain approximately 33,000 cells in total, using k=100 typically takes around 70 minutes on 10 threads. If your system has limited CPU resources, you can reduce the number of threads (`n_jobs`) or decrease the value of k (though it is recommended not to go below 100) to optimize resource usage and avoid overloading.

In [ ]:
# Step 3: Identify DSE genes
fdr_before_filtered, _ = se.SpaceExpress_DSE(embedding_before_filtered, adata_list, cell_type = 'Cell_class', k = 100, multi = True, group_id = [0,0,0,1,1,1])
summary_before_filtered = se.summary_DSE(fdr_before_filtered, threshold=0.001)
dse_genes_before_filtered = list(set([item for sublist in summary_before_filtered['DSE'].tolist() for item in sublist]))
non_dse_genes_before_filtered = [i for i in adata_list[0].var_names.tolist() if i not in dse_genes_before_filtered]

## 3) Round 2 of DSE genes detection

This round is recommended. It first removes the previously detected DSE genes and then retrains the SpaceExpress model. Removing strong DSE genes helps in creating a more stable coordinate system for spatial analysis.

In [ ]:
# Filter out the detected DSE genes from the dataset for retraining
adata_list_filtered = [adata[:, non_dse_genes_before_filtered] for adata in adata_list]
print('number of filtered genes: ',len(dse_genes_before_filtered))

In [ ]:
# Train the SpaceExpress model with the filtered dataset
se_embedding = se.train_SpaceExpress_multi(
    adata_list_filtered, 
    shortest_file_path_list, 
    epochs = 100
)
# Assign the embeddings to each sample in adata_list
for i in range(len(adata_list)):
    adata_list[i].obsm['SpaceExpress'] = se_embedding[i]

In [ ]:
# Perform DSE detection on the updated embeddings
df_fdr, adata_fdr = se.SpaceExpress_DSE(se_embedding, adata_list, cell_type = 'Cell_class', k = 100, multi = True, group_id = [0,0,0,1,1,1])

In [ ]:
# Optional: save the newly computed embedding and results.
# with open("Tutorial_data/Tutorial_2_data/result.pkl", "wb") as handle:
#     pickle.dump([se_embedding, adata_list, df_fdr, adata_fdr], handle)


In [ ]:
# Summary of DSE Results
se.summary_DSE(df_fdr)

In [ ]:
flat_fdr = df_fdr.unstack().reset_index()
flat_fdr.columns = ['Gene', 'Dimension', 'FDR']
top_10_fdr = flat_fdr.sort_values(by='FDR').head(10)
top_10_fdr

## Visualization of the DSE Result

- **plot_DSE**: This function visualizes the results of DSE analysis. By specifying the gene name and dimension of interest, `plot_DSE` displays the differential spatial expression patterns across naive and pup-exposed samples. The function also shows spline model fitting to highlight expression pattern differences between the samples.

In [ ]:
# Example usage of plot_DSE function to visualize DSE results for a specific gene and dimension
fig = se.plot_DSE(adata_fdr, df_fdr, file_list, gene_name = 'Oxt', dimension = 3, multi = True, group_id = [0,0,0,1,1,1])